# Qwen-Image 2.1 Generation with Optional Checkpoints and Multiple LoRAs

Run the cells in order with the project's Python 3.11 GPU kernel. `QwenImage21Pipeline` loads `Qwen/Qwen-Image-2.1`, optionally replacing its transformer with a local Qwen 2.1 `.safetensors` checkpoint. The supporting text encoder, processor, VAE, and scheduler still come from `model_id`.

Assets live under `Models/Model_Families/QWEN_2_1/`: checkpoints in `checkpoints/`, adapters in `loras/`. Absolute paths work too. `.env`, relative `CACHE_LOCATION`, and output paths resolve from the project root; Hub downloads share `CACHE_LOCATION/hub` (default `Models/hub`). Set `HUGGINGFACE_TOKEN` or `HF_TOKEN` in `.env` if needed.

References: [Qwen 2.1 model card](https://huggingface.co/Qwen/Qwen-Image-2.1), [Diffusers Qwen 2.1 pipeline and single-file loading](https://huggingface.co/docs/diffusers/en/api/pipelines/qwenimage21).


### 1. Check the project environment

Use the environment provisioned by `scripts/setup.sh --dev` and select its notebook kernel. This cell checks imports without upgrading the shared runtime or replacing the CUDA wheels. If dependencies are missing, use the project setup script before restarting the kernel.


In [ ]:
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import re
import secrets
from time import perf_counter
from uuid import uuid4

import diffusers
import peft
import torch
import transformers
from diffusers import QwenImage21Pipeline, QwenImage21Transformer2DModel
from dotenv import dotenv_values, load_dotenv
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
from IPython.display import display

print(f"PyTorch {torch.__version__} | Diffusers {diffusers.__version__}")
print(f"Transformers {transformers.__version__} | PEFT {peft.__version__}")
print("Qwen 2.1 pipeline, single-file transformer loader, and LoRA support imported.")


### 2. Configure the model source, prompts, and LoRA stack

`model_source = "auto"` uses `checkpoint_file` if it exists and otherwise loads the full Hugging Face model. `"hub"` always uses `model_id`; `"checkpoint"` requires the configured local file and fails if it is missing. Set `checkpoint_file = None` to use the Hub in auto mode. The supplied path selects your Big Love Qiara1 checkpoint when present. Auto mode only falls back for a missing file, never for an incompatible or corrupt checkpoint.

Add any number of Qwen **2.1** LoRAs to `loras`, each with a unique `name`, local `weights` path, independent `scale`, and `enabled` flag. The local Turbo adapter is shown disabled so it can be enabled deliberately with its matching sampling settings. Signed strengths are supported, including the negative Turbo ratios described by the Big Love checkpoint author. Add any required trigger words to the prompts. `use_loras = False`, an empty list, or disabling every entry restores the selected base/checkpoint without adapters.

Rerun this cell and all subsequent cells after edits. Strength changes reuse loaded adapters; changing the enabled file stack reloads adapters; switching model source reloads the pipeline.

The base model uses 40 steps with `true_cfg_scale = 1.0`. Negative prompting requires a nonempty `negative_prompt` and `true_cfg_scale > 1`. `sigmas = None` preserves the repository's schedule. For a Turbo model/checkpoint, use its matching repository/schedule and author instructions; reducing the step count alone does not configure Turbo. An explicit `sigmas` list overrides the repository schedule and determines the step count.


In [ ]:
# Find the project from either its root or a notebook's nested working directory.
project_root = next(
    (parent for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (parent / "Models" / "Model_Families").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook kernel inside the Image_Generator project.")
model_family = "QWEN_2_1"
family_dir = project_root / "Models" / "Model_Families" / model_family
checkpoint_root = family_dir / "checkpoints"
lora_root = family_dir / "loras"

def resolve_path(value, relative_to):
    """Resolve relative paths against an explicit root; preserve absolute paths."""
    path = Path(value).expanduser()
    return (path if path.is_absolute() else relative_to / path).resolve()


prompts = [
    "A cat holding a sign that says hello world",
    "A cinematic photograph of a mountain lake at sunrise, mist over the water, golden light",
    "A friendly robot tending a rooftop greenhouse, watercolor illustration, soft pastel colors",
    "A glowing bioluminescent forest at night, tiny fireflies, intricate fantasy concept art",
]

model_id = "Qwen/Qwen-Image-2.1"
model_source = "auto"          # "auto", "hub", or "checkpoint".
checkpoint_file = "Big_Love_Qwen_2/BigLoveQiara1_bf16.safetensors"  # None = Hub in auto mode.
height = 2048                 # Pixels; positive multiples of 32.
width = 2048
num_inference_steps = 40      # Base model default; custom sigmas override this.
true_cfg_scale = 1.0          # > 1 plus a nonempty negative_prompt enables CFG.
negative_prompt = ""
sigmas = None                 # Optional descending list in (0, 1]; omit terminal zero.
use_kv_cache = True
num_images_per_prompt = 1
seed = 42                     # None = random; integer = repeatable seed sequence.

use_loras = True
loras = [
    {
        "name": "turbo",
        "weights": "qwen21_turbo_rank256.safetensors",
        "scale": 0.3,
        "enabled": False,     # Enable with the author's matching sampling settings.
    },
    # Add more actual Qwen 2.1 LoRA files to combine adapters:
    # {"name": "style", "weights": "MyStyle/style.safetensors", "scale": 0.7, "enabled": True},
    # {"name": "detail", "weights": "MyDetail/detail.safetensors", "scale": 0.4, "enabled": True},
]

output_dir = "Images/QWEN_2_1"    # Relative to the project root, or absolute.
show_images = True


### 3. Validate settings and local assets

Validation finishes before downloading weights or changing an existing pipeline. Disabled LoRA entries are ignored. File fingerprints detect replacements at the same path without hashing large weight files.


In [ ]:
def finite_number(value):
    return (
        not isinstance(value, bool)
        and isinstance(value, (int, float))
        and math.isfinite(value)
    )


def local_weights(value, root, label):
    if not isinstance(value, (str, Path)) or not str(value).strip():
        raise ValueError(f"Set {label} to a local .safetensors file.")
    path = resolve_path(value, root)
    if path.suffix.lower() != ".safetensors":
        raise ValueError(f"{label} must be a .safetensors file.")
    if not path.is_file():
        raise FileNotFoundError(f"{label} not found: {path}")
    stat = path.stat()
    return path, (str(path), stat.st_size, stat.st_mtime_ns)


if not isinstance(prompts, (list, tuple)) or not prompts:
    raise ValueError("prompts must be a nonempty list of strings.")
if any(not isinstance(prompt, str) or not prompt.strip() for prompt in prompts):
    raise ValueError("Every prompt must be a nonempty string.")
unique_prompts = list(dict.fromkeys(prompts))
if not isinstance(model_id, str) or not model_id.strip():
    raise ValueError("model_id must identify the matching Qwen 2.1 component repository.")
for name, value in {"height": height, "width": width}.items():
    if type(value) is not int or value <= 0 or value % 32:
        raise ValueError(f"{name} must be a positive integer divisible by 32.")
for name, value in {
    "num_inference_steps": num_inference_steps,
    "num_images_per_prompt": num_images_per_prompt,
}.items():
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} must be a positive integer.")
if seed is not None and (type(seed) is not int or not 0 <= seed < 2**63):
    raise ValueError("seed must be None or an integer between 0 and 2**63 - 1.")
if not finite_number(true_cfg_scale) or true_cfg_scale < 1:
    raise ValueError("true_cfg_scale must be a finite number >= 1.")
if type(use_kv_cache) is not bool:
    raise ValueError("use_kv_cache must be True or False.")
if true_cfg_scale > 1 and (not isinstance(negative_prompt, str) or not negative_prompt.strip()):
    raise ValueError("Set a nonempty negative_prompt when true_cfg_scale > 1.")
if sigmas is not None:
    if not isinstance(sigmas, (list, tuple)) or not sigmas:
        raise ValueError("sigmas must be None or a nonempty descending list in (0, 1].")
    if any(not finite_number(value) or not 0 < value <= 1 for value in sigmas):
        raise ValueError("Each sigma must be finite and in (0, 1]; omit the terminal zero.")
    if any(left <= right for left, right in zip(sigmas, sigmas[1:])):
        raise ValueError("sigmas must be strictly descending.")
if not isinstance(negative_prompt, str):
    raise ValueError("negative_prompt must be a string; use an empty string to leave it blank.")
if type(show_images) is not bool or type(use_loras) is not bool:
    raise ValueError("show_images and use_loras must be True or False.")
if not isinstance(output_dir, (str, Path)) or not str(output_dir).strip():
    raise ValueError("output_dir must be a nonempty folder path.")
if not isinstance(loras, (list, tuple)):
    raise ValueError("loras must be a list of LoRA configuration dictionaries.")

if model_source not in ("auto", "hub", "checkpoint"):
    raise ValueError('model_source must be "auto", "hub", or "checkpoint".')
checkpoint_path = checkpoint_key = None
if model_source != "hub":
    if checkpoint_file is not None and (not isinstance(checkpoint_file, (str, Path)) or not str(checkpoint_file).strip()):
        raise ValueError("checkpoint_file must be None or a nonempty .safetensors path.")
    if checkpoint_file is not None:
        candidate = resolve_path(checkpoint_file, checkpoint_root)
        if candidate.suffix.lower() != ".safetensors":
            raise ValueError("Checkpoint must be a .safetensors file.")
        if model_source == "checkpoint" or candidate.exists():
            checkpoint_path, checkpoint_key = local_weights(checkpoint_file, checkpoint_root, "Checkpoint")
        else:
            print(f"Optional checkpoint not found: {candidate}; using {model_id}.")
    elif model_source == "checkpoint":
        raise ValueError('Set checkpoint_file when model_source is "checkpoint".')
resolved_model_source = "checkpoint" if checkpoint_path is not None else "hub"
active_loras = []
adapter_names_seen = set()
if use_loras:
    for index, item in enumerate(loras):
        if not isinstance(item, dict):
            raise ValueError(f"LoRA entry {index + 1} must be a dictionary.")
        enabled = item.get("enabled", True)
        if type(enabled) is not bool:
            raise ValueError(f"LoRA entry {index + 1}: enabled must be True or False.")
        if not enabled:
            continue
        name = item.get("name")
        if not isinstance(name, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", name):
            raise ValueError(f"LoRA entry {index + 1}: name must contain only letters, digits, _ or -.")
        if name in adapter_names_seen:
            raise ValueError(f"Duplicate enabled LoRA name: {name!r}.")
        scale = item.get("scale", 1.0)
        if not finite_number(scale):
            raise ValueError(f"LoRA {name!r}: scale must be a finite number.")
        path, fingerprint = local_weights(item.get("weights"), lora_root, f"LoRA {name!r}")
        active_loras.append({"name": name, "path": path, "scale": float(scale), "fingerprint": fingerprint})
        adapter_names_seen.add(name)

# Exclude strengths so changing only a scale does not reload the adapter files.
lora_stack_key = tuple((item["name"], item["fingerprint"]) for item in active_loras)
output_root = resolve_path(output_dir, project_root)
print(f"Validated {len(unique_prompts)} prompt(s), {resolved_model_source} model, and {len(active_loras)} active LoRA(s).")


### 4. Load or reuse the Qwen 2.1 pipeline

Load in BF16 on CUDA, following the project's Thor GPU workflows. Hub mode loads the complete pipeline. Checkpoint mode uses `QwenImage21Transformer2DModel.from_single_file` and supplies that transformer to the pipeline, avoiding a download of the base transformer. Only Qwen 2.1 transformer checkpoints are supported here; use `model_id` for a complete custom Diffusers repository.

The loading cell rereads `.env` credentials on each run. For HTTP 401/403, verify that the token's account has access to `model_id`. A small configuration download checks access before releasing the existing pipeline. The checkpoint fingerprint detects file replacement at the same path.


In [ ]:
load_dotenv(project_root / ".env")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the project's CUDA-enabled Python kernel.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("This notebook requires a CUDA device with BF16 support.")

device = torch.device("cuda:0")
dtype = torch.bfloat16
gib = 1024**3
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
print(f"GPU: {torch.cuda.get_device_name(device)} | CUDA: {torch.version.cuda}")
print(f"CUDA memory: {free_bytes / gib:.1f} GiB free / {total_bytes / gib:.1f} GiB total")

cache_root = resolve_path(os.getenv("CACHE_LOCATION") or "Models", project_root)
cache_dir = cache_root / "hub"
cache_dir.mkdir(parents=True, exist_ok=True)
# Re-read the token from .env on every run, including after editing credentials.
# Do not print tokens. None lets the Hub use its saved login when no token is set.
env_values = dotenv_values(project_root / ".env")
hf_token = (
    env_values.get("HUGGINGFACE_TOKEN") or env_values.get("HF_TOKEN")
    or os.getenv("HUGGINGFACE_TOKEN") or os.getenv("HF_TOKEN") or None
)
load_options = {
    "cache_dir": str(cache_dir),
    "token": hf_token,
    "dtype": dtype,
    "low_cpu_mem_usage": True,
}
pipeline_config = (model_id, checkpoint_key, str(cache_dir), dtype, str(device))
if "pipe" not in globals() or globals().get("_loaded_qwen21_config") != pipeline_config:
    # Check repository access before releasing an existing pipeline or loading weights.
    if not Path(model_id).is_dir():
        try:
            hf_hub_download(
                repo_id=model_id,
                filename="transformer/config.json",
                cache_dir=str(cache_dir),
                token=hf_token,
            )
        except HfHubHTTPError as exc:
            status = getattr(getattr(exc, "response", None), "status_code", None)
            if isinstance(exc, GatedRepoError) or status in (401, 403):
                raise RuntimeError(
                    f"Hugging Face denied access to {model_id} (HTTP {status}). "
                    "The pipeline needs the repository config and supporting components; "
                    "a custom transformer checkpoint does not include these assets. "
                    f"Open https://huggingface.co/{model_id}, sign in with the account "
                    "that owns your token, and request/accept model access. Once approved, "
                    "ensure the token has read access to this gated repository. "
                    "Set HUGGINGFACE_TOKEN or HF_TOKEN in the project .env, then rerun this cell. "
                    "If access was just granted, allow time for it to take effect."
                ) from None
            raise
    _loaded_qwen21_config = None
    if "pipe" in globals():
        del pipe
    gc.collect()
    torch.cuda.empty_cache()
    pipeline_options = {}
    if checkpoint_path is not None:
        transformer = QwenImage21Transformer2DModel.from_single_file(
            str(checkpoint_path), config=model_id, subfolder="transformer", **load_options,
        ).to(device)
        pipeline_options["transformer"] = transformer
        del transformer
    try:
        pipe = QwenImage21Pipeline.from_pretrained(
            model_id,
            device_map="cuda",
            use_safetensors=True,
            **pipeline_options,
            **load_options,
        )
    finally:
        # The pipeline owns the transformer after success; release it on failure too.
        pipeline_options.clear()
    _loaded_qwen21_config = pipeline_config
else:
    print("Reusing the loaded Qwen 2.1 pipeline.")
print(f"Model source: {resolved_model_source} | {checkpoint_path or model_id}")


### 5. Apply the selected LoRAs together

Each enabled file loads under its own adapter name; `set_adapters` combines their independent strengths. Adapters remain separate from base weights, so disabling or removing the stack restores the selected model. Only use adapters trained for Qwen 2.1. A failed adapter load clears the partial stack before raising an error.


In [ ]:
try:
    if getattr(pipe, "_notebook_qwen21_lora_key", None) != lora_stack_key:
        pipe.unload_lora_weights()
        pipe._notebook_qwen21_lora_key = None
        for item in active_loras:
            print(f"Loading LoRA {item['name']}: {item['path'].name}")
            pipe.load_lora_weights(
                str(item["path"].parent),
                weight_name=item["path"].name,
                adapter_name=item["name"],
                local_files_only=True,
                use_safetensors=True,
            )
    if active_loras:
        adapter_names = [item["name"] for item in active_loras]
        adapter_weights = [item["scale"] for item in active_loras]
        pipe.set_adapters(adapter_names, adapter_weights=adapter_weights)
        pipe.set_lora_device(adapter_names=adapter_names, device=device)
        for item in active_loras:
            print(f"LoRA: {item['name']} | Strength: {item['scale']}")
    else:
        print("No active LoRAs; using the selected model alone.")
    pipe._notebook_qwen21_lora_key = lora_stack_key
except Exception as exc:
    pipe._notebook_qwen21_lora_key = None
    pipe.unload_lora_weights()
    raise RuntimeError(
        "Could not configure the Qwen 2.1 LoRA stack; adapters were cleared. "
        "Check each enabled file's Qwen 2.1 compatibility and the error below."
    ) from exc

lora_settings = [
    {"name": item["name"], "weights": str(item["path"]), "scale": item["scale"]}
    for item in active_loras
]
for name in ("transformer", "text_encoder", "vae"):
    component = getattr(pipe, name)
    print(f"{name}: {component.device}, {component.dtype}")
del component


### 6. Generate and save images

Prompts run sequentially with `num_images_per_prompt` images per batch. PNGs and JSON sidecars go into prompt-specific folders under `Images/QWEN_2_1`. Metadata includes the resolved model source, optional checkpoint fingerprint, full active LoRA stack, seeds, requested/effective sampling schedule, runtime versions, and batch duration. Unique filenames preserve earlier runs. Keep the runtime, hardware, settings, and adapter stack fixed when comparing seeds.


In [ ]:
output_root.mkdir(parents=True, exist_ok=True)
saved_paths = []
torch.cuda.reset_peak_memory_stats(device)

# Process prompts in sequence; num_images_per_prompt controls the batch size.
with torch.inference_mode():
    for prompt_index, prompt in enumerate(unique_prompts):
        # The hash distinguishes different prompts with the same sanitized name.
        slug = re.sub(r"[^a-z0-9]+", "-", prompt.lower()).strip("-")[:60].rstrip("-") or "prompt"
        prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]
        prompt_dir = output_root / f"{slug}-{prompt_hash}"
        prompt_dir.mkdir(parents=True, exist_ok=True)
        (prompt_dir / "prompt.txt").write_text(prompt, encoding="utf-8")

        # Give each image its own recorded seed, including when seeds are random.
        seeds = [
            secrets.randbelow(2**63) if seed is None
            else (seed + prompt_index * num_images_per_prompt + image_index) % (2**63)
            for image_index in range(num_images_per_prompt)
        ]
        generators = [torch.Generator(device=device).manual_seed(value) for value in seeds]
        generation_settings = {
            "height": height,
            "width": width,
            "num_inference_steps": num_inference_steps,
            "true_cfg_scale": true_cfg_scale,
            "num_images_per_prompt": num_images_per_prompt,
            "negative_prompt": negative_prompt if true_cfg_scale > 1 else None,
            "sigmas": list(sigmas) if sigmas is not None else None,
            "use_kv_cache": use_kv_cache,
        }
        print(f"Prompt {prompt_index + 1}/{len(unique_prompts)}: {prompt}")
        torch.cuda.synchronize(device)
        start = perf_counter()
        images = pipe(prompt=prompt, generator=generators, **generation_settings).images
        torch.cuda.synchronize(device)
        elapsed = perf_counter() - start

        for image, image_seed in zip(images, seeds, strict=True):
            # A fresh UUID preserves prior images, even when rerunning the same seed.
            output_path = prompt_dir / f"qwen21-seed-{image_seed}-{uuid4().hex}.png"
            image.save(output_path)
            metadata = {
                "prompt": prompt,
                "model_id": model_id,
                "model_source": resolved_model_source,
                "checkpoint": str(checkpoint_path) if checkpoint_path is not None else None,
                "model_family": model_family,
                "checkpoint_fingerprint": checkpoint_key,
                "loras": [dict(item) for item in lora_settings],
                "lora_fingerprints": [item["fingerprint"] for item in active_loras],
                "seed": image_seed,
                "dtype": str(dtype),
                "device": str(device),
                "torch_version": torch.__version__,
                "diffusers_version": diffusers.__version__,
                "transformers_version": transformers.__version__,
                "peft_version": peft.__version__,
                "effective_sample_sigmas": list(sigmas) if sigmas is not None else pipe.config.get("sample_sigmas"),
                "effective_num_inference_steps": len(sigmas) if sigmas is not None else (
                    len(pipe.config.sample_sigmas) if pipe.config.get("sample_sigmas") else num_inference_steps
                ),
                "scheduler_class": type(pipe.scheduler).__name__,
                "scheduler_config": dict(pipe.scheduler.config),
                "batch_generation_seconds": elapsed,
                **generation_settings,
            }
            output_path.with_suffix(".json").write_text(
                json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
            )
            saved_paths.append(output_path)
            print(f"Saved {output_path}")
            if show_images:
                display(image)
        print(f"Generation: {elapsed:.1f}s for {len(images)} image(s)")

print(f"Saved {len(saved_paths)} images across {len(unique_prompts)} prompt folders.")
print(f"Peak PyTorch CUDA allocation: {torch.cuda.max_memory_allocated(device) / gib:.1f} GiB")
